# Variance Partitioning — Semantic vs Control Features
Unique variance explained by LLaMA semantics vs lexical/syntactic/acoustic controls, per neuron.

In [ ]:
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)

VP_DIR = '/scratch/aniluchavez/ConvoDATAS/VPResults'

dfs = []
for f in sorted(glob.glob(os.path.join(VP_DIR, '*_VP.pkl'))):
    with open(f, 'rb') as fh:
        dfs.append(pickle.load(fh))
data = pd.concat(dfs, ignore_index=True)

# clip extreme outliers (numerical artifacts from near-zero ll_null)
for col in ['unique_semantic', 'unique_controls', 'shared', 'r2_semantic', 'r2_controls', 'r2_full']:
    p1  = data[col].quantile(0.01)
    p99 = data[col].quantile(0.99)
    data[col] = data[col].clip(lower=p1, upper=p99)

data['pid_short'] = data['patient'].str[:5]
print(f'Loaded {data["patient"].nunique()} patients, {len(data)} neurons')
print(data.groupby(['region','condition'])['neuron_idx'].count())

## Unique Semantic Variance — Overall Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Unique Semantic Variance (R²_full − R²_controls)', fontsize=14, fontweight='bold')

regions    = ['hippocampus', 'ACC']
conditions = ['self', 'other']
cond_labels = {'self': 'Speaking (self)', 'other': 'Listening (other)'}
colors = {'hippocampus': 'steelblue', 'ACC': 'coral'}

for row, region in enumerate(regions):
    for col, cond in enumerate(conditions):
        ax  = axes[row, col]
        sub = data[(data['region'] == region) & (data['condition'] == cond)]['unique_semantic'].dropna()

        ax.hist(sub, bins=50, color=colors[region], alpha=0.75, edgecolor='white')
        ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
        ax.axvline(sub.median(), color='darkred', linewidth=1.5, label=f'median={sub.median():.4f}')
        pct_pos = (sub > 0).mean() * 100
        ax.set_title(f'{region.upper()} — {cond_labels[cond]}\n'
                     f'n={len(sub)}  {pct_pos:.0f}% > 0')
        ax.set_xlabel('Unique semantic R²')
        ax.set_ylabel('Neuron count')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VP_DIR, 'VP_unique_semantic_dist.png'), dpi=150, bbox_inches='tight')
plt.show()

## Semantic vs Control Variance — Scatter

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))
fig.suptitle('Unique Semantic vs Unique Control Variance', fontsize=14, fontweight='bold')

for row, region in enumerate(regions):
    for col, cond in enumerate(conditions):
        ax  = axes[row, col]
        sub = data[(data['region'] == region) & (data['condition'] == cond)].dropna(
                    subset=['unique_semantic', 'unique_controls'])

        ax.scatter(sub['unique_controls'], sub['unique_semantic'],
                   alpha=0.35, s=12, color=colors[region])
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
        ax.axvline(0, color='black', linewidth=0.8, linestyle='--')

        # highlight top semantic neurons
        top = sub.nlargest(10, 'unique_semantic')
        ax.scatter(top['unique_controls'], top['unique_semantic'],
                   s=40, color='darkred', zorder=5, label='top 10 semantic')

        ax.set_xlabel('Unique control R²')
        ax.set_ylabel('Unique semantic R²')
        ax.set_title(f'{region.upper()} — {cond_labels[cond]}  (n={len(sub)})')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VP_DIR, 'VP_sem_vs_ctrl_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

## VP Venn — Average Variance Partitioning by Region & Condition

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Mean Variance Partitioning (clipped at p1/p99)', fontsize=14, fontweight='bold')

bar_colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756']

for row, region in enumerate(regions):
    for col, cond in enumerate(conditions):
        ax  = axes[row, col]
        sub = data[(data['region'] == region) & (data['condition'] == cond)].dropna(
                    subset=['unique_semantic', 'unique_controls', 'shared', 'r2_full'])

        means = {
            'Unique\nSemantic':  sub['unique_semantic'].mean(),
            'Unique\nControls':  sub['unique_controls'].mean(),
            'Shared':            sub['shared'].mean(),
            'Total R²\n(Full)':  sub['r2_full'].mean(),
        }
        bars = ax.bar(means.keys(), means.values(), color=bar_colors, alpha=0.85, edgecolor='white')
        ax.axhline(0, color='black', linewidth=0.8)
        for bar, val in zip(bars, means.values()):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.0003,
                    f'{val:.4f}', ha='center', va='bottom', fontsize=8)

        ax.set_title(f'{region.upper()} — {cond_labels[cond]}  (n={len(sub)})')
        ax.set_ylabel('Mean R²')

plt.tight_layout()
plt.savefig(os.path.join(VP_DIR, 'VP_mean_partitioning.png'), dpi=150, bbox_inches='tight')
plt.show()

## Unique Semantic Variance by Patient (violin)

In [ ]:
for cond in conditions:
    sub = data[(data['region'] == 'hippocampus') & (data['condition'] == cond)].dropna(subset=['unique_semantic'])
    if sub.empty:
        continue
    order = sorted(sub['pid_short'].unique())

    fig, ax = plt.subplots(figsize=(16, 5))
    sns.violinplot(data=sub, x='pid_short', y='unique_semantic', order=order,
                   palette='husl', inner='box', cut=0, ax=ax)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_title(f'Hippocampus — {cond_labels[cond]}\nUnique Semantic Variance per Patient',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Patient')
    ax.set_ylabel('Unique semantic R²')
    plt.tight_layout()
    plt.savefig(os.path.join(VP_DIR, f'VP_per_patient_hippo_{cond}.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Top Semantic Neurons — Ranked

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Neurons Ranked by Unique Semantic Variance', fontsize=14, fontweight='bold')

for row, region in enumerate(regions):
    for col, cond in enumerate(conditions):
        ax  = axes[row, col]
        sub = data[(data['region'] == region) & (data['condition'] == cond)].dropna(
                    subset=['unique_semantic']).sort_values('unique_semantic', ascending=False).reset_index(drop=True)

        ax.bar(range(len(sub)), sub['unique_semantic'], color=colors[region], alpha=0.75, width=1.0)
        ax.axhline(0, color='black', linewidth=0.8)
        n_pos = (sub['unique_semantic'] > 0).sum()
        ax.set_title(f'{region.upper()} — {cond_labels[cond]}\n{n_pos}/{len(sub)} neurons > 0')
        ax.set_xlabel('Neuron rank')
        ax.set_ylabel('Unique semantic R²')
        ax.tick_params(axis='x', labelbottom=False)

plt.tight_layout()
plt.savefig(os.path.join(VP_DIR, 'VP_ranked_neurons.png'), dpi=150, bbox_inches='tight')
plt.show()

## Self vs Other — Direct Comparison

In [ ]:
# neurons that appear in both self and other (same patient, region, neuron_idx)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Unique Semantic Variance: Speaking vs Listening', fontsize=14, fontweight='bold')

for ax, region in zip(axes, regions):
    self_df  = data[(data['region']==region) & (data['condition']=='self')][['patient','neuron_idx','unique_semantic']].rename(columns={'unique_semantic':'self'})
    other_df = data[(data['region']==region) & (data['condition']=='other')][['patient','neuron_idx','unique_semantic']].rename(columns={'unique_semantic':'other'})
    both = self_df.merge(other_df, on=['patient','neuron_idx']).dropna()

    ax.scatter(both['self'], both['other'], alpha=0.3, s=12, color=colors[region])
    lim = max(abs(both[['self','other']].values.ravel().max()),
              abs(both[['self','other']].values.ravel().min())) * 1.1
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=0.8, label='y=x')
    ax.axhline(0, color='grey', linewidth=0.5)
    ax.axvline(0, color='grey', linewidth=0.5)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel('Unique semantic R² (speaking)')
    ax.set_ylabel('Unique semantic R² (listening)')
    ax.set_title(f'{region.upper()}  (n={len(both)} matched neurons)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(VP_DIR, 'VP_self_vs_other.png'), dpi=150, bbox_inches='tight')
plt.show()